### Approach

Regularly back up checkpoint. When it becomes corrupt, replace it with backup.
Then:

- get the latest completed microbatch from restored checkpoint
- get the target table Delta commit version written by that microbatch
- restore target table back to that Delta commit version

Resume stream.

Don't need to change code or deduplicate.

Requirements: source data must still exist for reprocessing

### Demo

#### Clean up previous output

In [0]:
%fs rm -r /Volumes/alexn/default/v/checkpoints/trips/


res4: Boolean = true

In [0]:
%fs rm -r /Volumes/alexn/default/v/checkpoint-backups/trips/

res5: Boolean = true

In [0]:
%fs rm -r /Volumes/alexn/default/v/trips

res6: Boolean = true

#### Start stream, backup checkpoint and experience corruption

In [0]:
def run_stream():
  df = spark.readStream.option("maxFilesPerTrigger", 1).table("alexn.default.trips_p10")

  query = (
      df.writeStream
      .format("delta")
      .option("checkpointLocation", "/Volumes/alexn/default/v/checkpoints/trips/")
      .start("/Volumes/alexn/default/v/trips")
  )

In [0]:
run_stream()

In [0]:
dbutils.fs.cp("/Volumes/alexn/default/v/checkpoints/trips/", "/Volumes/alexn/default/v/checkpoint-backups/trips/", recurse=True)

True

(stop the stream)

In [0]:
%fs ls /Volumes/alexn/default/v/checkpoint-backups/trips/commits/

path,name,size,modificationTime
dbfs:/Volumes/alexn/default/v/checkpoint-backups/trips/commits/0,0,29,1760102491000
dbfs:/Volumes/alexn/default/v/checkpoint-backups/trips/commits/1,1,29,1760102492000
dbfs:/Volumes/alexn/default/v/checkpoint-backups/trips/commits/2,2,29,1760102492000
dbfs:/Volumes/alexn/default/v/checkpoint-backups/trips/commits/__tmp_path_dir/,__tmp_path_dir/,0,1760102492000


In [0]:
%fs ls /Volumes/alexn/default/v/checkpoints/trips/commits/

path,name,size,modificationTime
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/0,0,29,1760102484000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/1,1,29,1760102486000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/2,2,29,1760102489000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/3,3,29,1760102491000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/4,4,29,1760102494000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/__tmp_path_dir/,__tmp_path_dir/,0,1760102484000


In [0]:
%fs ls /Volumes/alexn/default/v/checkpoints/trips/offsets/

path,name,size,modificationTime
dbfs:/Volumes/alexn/default/v/checkpoints/trips/offsets/0,0,1192,1760102480000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/offsets/1,1,1192,1760102484000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/offsets/2,2,1192,1760102487000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/offsets/3,3,1192,1760102489000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/offsets/4,4,1192,1760102492000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/offsets/5,5,1192,1760102494000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/offsets/__tmp_path_dir/,__tmp_path_dir/,0,1760102480000


In [0]:
%sh echo "invalid content" > /tmp/junkfile

In [0]:
%sh cp /tmp/junkfile /Volumes/alexn/default/v/checkpoints/trips/offsets/5

In [0]:
run_stream()

#### Recover from backup

In [0]:
%fs mv -r /Volumes/alexn/default/v/checkpoints/trips /Volumes/alexn/default/v/corrupt-checkpoints/trips

res10: Boolean = true

In [0]:
%fs cp -r  /Volumes/alexn/default/v/checkpoint-backups/trips  /Volumes/alexn/default/v/checkpoints/trips 

res12: Boolean = true

In [0]:
%fs ls /Volumes/alexn/default/v/checkpoints/trips/commits/

path,name,size,modificationTime
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/0,0,29,1760103318000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/1,1,29,1760103318000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/2,2,29,1760103319000
dbfs:/Volumes/alexn/default/v/checkpoints/trips/commits/__tmp_path_dir/,__tmp_path_dir/,0,1760103319000


In [0]:
sorted(dbutils.fs.ls("/Volumes/alexn/default/v/checkpoints/trips/commits"), key=lambda x: x.modificationTime, reverse=True)[0].name

'2'

In [0]:
%sql
desc history delta.`/Volumes/alexn/default/v/trips/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
6,2025-10-10T13:21:36Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 5, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,5,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56238, numOutputRows -> 2194, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
5,2025-10-10T13:21:33Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 4, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,4,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56185, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
4,2025-10-10T13:21:30Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 3, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,3,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56167, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
3,2025-10-10T13:21:28Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 2, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,2,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56123, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
2,2025-10-10T13:21:26Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 1, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,1,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56275, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
1,2025-10-10T13:21:23Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 0, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,0,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56228, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
0,2025-10-10T13:21:21Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> -1, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.12


In [0]:
%sql

select version
from (describe history delta.`/Volumes/alexn/default/v/trips/`)
where operationParameters.epochId=2
--and operationParameters.queryId='4cb43030-1291-4629-9aed-090dd730f192'   -- in case you have multiple stream queries writing to the same table

version
3


In [0]:
%sql

RESTORE TABLE delta.`/Volumes/alexn/default/v/trips/` TO VERSION AS OF 3;

table_size_after_restore,num_of_files_after_restore,num_removed_files,num_restored_files,removed_files_size,restored_files_size
168626,3,3,0,168590,0


In [0]:
%sql
desc history delta.`/Volumes/alexn/default/v/trips/`

version,timestamp,userId,userName,operation,operationParameters,job,notebook,clusterId,readVersion,isolationLevel,isBlindAppend,operationMetrics,userMetadata,engineInfo
7,2025-10-10T13:41:51Z,478828478277646,alex.nastetsky@databricks.com,RESTORE,"Map(timestamp -> null, version -> 3)",null,List(1011186802938259),0918-131710-dfwhhk1o,6,Serializable,false,"Map(numRestoredFiles -> 0, removedFilesSize -> 168590, numRemovedFiles -> 3, restoredFilesSize -> 0, numOfFilesAfterRestore -> 3, tableSizeAfterRestore -> 168626)",null,Databricks-Runtime/16.4.x-scala2.12
6,2025-10-10T13:21:36Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 5, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,5,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56238, numOutputRows -> 2194, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
5,2025-10-10T13:21:33Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 4, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,4,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56185, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
4,2025-10-10T13:21:30Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 3, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,3,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56167, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
3,2025-10-10T13:21:28Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 2, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,2,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56123, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
2,2025-10-10T13:21:26Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 1, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,1,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56275, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
1,2025-10-10T13:21:23Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> 0, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,0,WriteSerializable,true,"Map(numAddedFiles -> 1, numOutputBytes -> 56228, numOutputRows -> 2193, numRemovedFiles -> 0)",null,Databricks-Runtime/16.4.x-scala2.12
0,2025-10-10T13:21:21Z,478828478277646,alex.nastetsky@databricks.com,STREAMING UPDATE,"Map(epochId -> -1, outputMode -> Append, queryId -> 4cb43030-1291-4629-9aed-090dd730f192, statsOnLoad -> false)",null,List(1011186802938259),0918-131710-dfwhhk1o,null,WriteSerializable,true,Map(),null,Databricks-Runtime/16.4.x-scala2.12


In [0]:
run_stream()

#### Validation

In [0]:
%sql
select count(*) from alexn.default.trips_p10

count(1)
153524


In [0]:
%sql
select count(*) from delta.`/Volumes/alexn/default/v/trips/`

count(1)
153524
